# 06 — Progressives Fine-Tuning

Entspricht Kapitel 5, Abschnitt *Progressives Fine-Tuning*.

Prueft, ob die **Reihenfolge** der Anpassungsschritte eine Rolle
spielt. Fuer die Insole ist das unmittelbar relevant: Soll das Modell
direkt von SisFall auf die Fussdaten angepasst werden, oder lohnt ein
Zwischenschritt ueber einen weiteren Sturzdatensatz?

Drei Strategien, identischer Testsatz:

| | Strategie |
|---|---|
| A | Zero-Shot (SisFall unveraendert) |
| B | Direkt: SisFall → Ziel |
| C | Progressiv: SisFall → KFall → Ziel |

> **Korrektur:** KFall wird mit der berichtigten Sturzgrenze
> (T20–T34) geladen. Fruehere Laeufe dieses Experiments verwendeten
> T22–T36 und sind nicht mehr gueltig.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Gemeinsame Bibliothek holen (einzige Quelle fuer alle Loader)
!wget -q https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py -O /content/fall_lib.py
sys.path.insert(0, "/content")
import fall_lib as fl

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"      # einheitlicher Modellpfad

print("Konfiguration:", fl.default_config())

Mounted at /content/drive
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


In [2]:
def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte.
    Drive ist fuer tausende Kleindateien zu langsam."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

In [3]:
from tensorflow import keras
basis = keras.models.load_model(MODEL, compile=False)
basis.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
run = fl.new_run("06_progressiv")

ZIEL_POSITION = "ANKLE"

kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                   label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"))
uma = fl.load_umafall(
    hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
         "/content/UMAFall"), position=ZIEL_POSITION)

X_kf, y_kf, _, _ = fl.build_windows(kf, mode="standard")
X_kf = fl.znorm(X_kf)

X_um, y_um, g_um, _ = fl.build_windows(uma, mode="standard")
X_um = fl.znorm(X_um)
tr, te = fl.subject_split(g_um, y_um, test_size=0.25)
print(f"\nZiel-Testsatz: {len(te)} Fenster, "
      f"{int(y_um[te].sum())} Stuerze, "
      f"{len(np.unique(g_um[te]))} Probanden")

[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/06_progressiv_2026-08-26_1149
[ok] KFALL/sensor_data_new kopiert in 137s
[ok] KFALL/label_data_new kopiert in 12s
[KFall] 5075 Trials (2346 Sturz, 2729 ADL), 32 Probanden
[KFall] Sturz-Tasks T20-T34 | 2346 Aufprallzeitpunkte aus Labeldateien
[ok] UMAFall Fall Detection/UMAFall_Dataset_corrected kopiert in 16s
[UMAFall ANKLE] 643 Trials (157 Sturz, 486 ADL), 19 Probanden

Ziel-Testsatz: 1979 Fenster, 36 Stuerze, 5 Probanden


In [4]:
print(f"\nVergleich auf identischem Testsatz "
      f"(UMAFall {ZIEL_POSITION})\n")
erg = {}

erg["A_zero_shot"] = fl.evaluate(basis, X_um[te], y_um[te],
                                 "A  Zero-Shot")

m_direkt = fl.finetune(basis, X_um[tr], y_um[tr])
erg["B_direkt"] = fl.evaluate(m_direkt, X_um[te], y_um[te],
                              "B  Direkt")

m_kf = fl.finetune(basis, X_kf, y_kf)
erg["C1_nach_kfall"] = fl.evaluate(m_kf, X_um[te], y_um[te],
                                   "C1 nach KFall (ZS)")

m_prog = fl.finetune(m_kf, X_um[tr], y_um[tr])
erg["C2_progressiv"] = fl.evaluate(m_prog, X_um[te], y_um[te],
                                   "C2 Progressiv")


Vergleich auf identischem Testsatz (UMAFall ANKLE)

  A  Zero-Shot                Se=0.194  Sp=0.972  Acc=0.958  FP=   55  FN=  29
  B  Direkt                   Se=0.278  Sp=0.966  Acc=0.954  FP=   66  FN=  26


  C1 nach KFall (ZS)          Se=0.139  Sp=0.986  Acc=0.971  FP=   27  FN=  31


  C2 Progressiv               Se=0.333  Sp=0.960  Acc=0.949  FP=   77  FN=  24


In [5]:
d = erg["C2_progressiv"]["Se"] - erg["B_direkt"]["Se"]
print(f"\nProgressiv gegenueber direkt: {d*100:+.1f} Prozentpunkte "
      f"Sensitivitaet")
print(f"Spezifitaet: {erg['B_direkt']['Sp']:.3f} -> "
      f"{erg['C2_progressiv']['Sp']:.3f}")

if erg["C1_nach_kfall"]["Se"] < erg["A_zero_shot"]["Se"] and d > 0.02:
    print("\nDie Zwischenstufe verschlechtert die unmittelbare "
          "Leistung, verbessert aber die Anpassbarkeit im "
          "darauffolgenden Schritt.")

fl.save_results(run, {"strategien": erg,
                      "delta_progressiv_direkt": round(float(d), 4),
                      "ziel_position": ZIEL_POSITION},
                config=fl.default_config())


Progressiv gegenueber direkt: +5.5 Prozentpunkte Sensitivitaet
Spezifitaet: 0.966 -> 0.960

Die Zwischenstufe verschlechtert die unmittelbare Leistung, verbessert aber die Anpassbarkeit im darauffolgenden Schritt.
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/06_progressiv_2026-08-26_1149/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/06_progressiv_2026-08-26_1149/results.json')